# Question 1: River Forecasting Proposal

**Goal:** every 10 minutes, forecast the water level 1 hour ahead at main-river stations S1–S4.

**Design Idea:** a graph of the river (reservoir → S1 → S2 → S3 → S4, with T1 joining at S2). A GRU summarises each node's last 6 hours, message passing carries information downstream, and an output head predicts all six 10-minute steps up to +60 min at once.

## 1. Architecture diagram

![Architecture](architecture.png)

- **Nodes (N = 6):** Res (reservoir release point), S1, T1, S2, S3, S4. The unmonitored tributary U has no data, so it is not a node.
- **Alignment and reshaping:**
  - All series are put on the same 10-minute grid.
  - Rain gauges are not river nodes, so the fixed map A (6 × 2) assigns their rainfall to nodes.
  - Recorded releases go into the Res node's release channel.
  - Announced future releases cover a different time span (the next hour), so they are encoded separately and added to the Res node's state.
  - The GRU runs on each node separately (reshape to B·6 sequences), then the result is reshaped back to (B, 6, 64) for the graph layers.
- **Output:** the head gives (B, 4, 6), which is transposed to (B, 6, 4). The service publishes the +60 min slice.

## 2. Assumptions

**Supplied facts (not assumptions):**
- 10-minute levels at S1–S4 and T1.
- R1 and R2 rainfall with timestamps and locations.
- Recorded releases, and future schedules with announcement and effective times.
- Station locations, connectivity and river distances.
- Exact travel times and the unmonitored tributary's inflow are not supplied.
- Future rain and future levels are not available at issuance.

# Assumptions:
- A1: We have access to significant historical data, enought to create a reasonable train/val/test set.
- A2: the time is synced across all the sensors, if not the data cannot be stacked uniformly. 
- A3: There are missing values and delay is expected for the readings i.e some might arrive late. Needs to be catered in the design. 
- A4: Spatially, water only flows downstream and doesnot affect upstream flow. 
- A5: Each rain gauge represents the nearby area of the river it measures.   

## 3. Forecasting task and data specification

- **Issuance time t0:** every 10 minutes (on the 10-min grid). The forecast uses only data available at t0.
- **History window:** L = 36 steps (6 h), covering t0 − 350 min … t0 (A7).
- **Horizon:** 6 steps, t0 + 10 … t0 + 60 min. All six are predicted; the service uses t0 + 60. The extra five are cheap and give more training signal.
- **Outputs:** water level at S1, S2, S3, S4.

| Item | Shape | Axes | Time coverage |
|---|---|---|---|
| Water levels | (B, 36, 5) | batch, time step, station (S1–S4, T1) | t0 − 350 … t0 |
| Missing mask for levels | (B, 36, 5) | same as above (1 = observed) | t0 − 350 … t0 |
| Rainfall | (B, 36, 2) | batch, time step, gauge (R1, R2) | 10-min totals ending t0 − 350 … t0 |
| Recorded releases | (B, 36, 1) | batch, time step, reservoir | t0 − 350 … t0 |
| Announced future releases | (B, 6, 2) | batch, future step, [release value, announced flag] | t0 + 10 … t0 + 60, only if announced ≤ t0 |
| River graph | (6, 6) adjacency + (6, 6) distances | node × node | static |
| Gauge-to-node map A | (6, 2) | node × gauge | static |
| Model input X | (B, 36, 6, 4) | batch, time step, node, feature [level, mask, rain, release] | t0 − 350 … t0 |
| Output Ŷ | (B, 6, 4) | batch, horizon (+10 … +60 min), station (S1–S4) | t0 + 10 … t0 + 60 |
| Target Y (training only) | (B, 6, 4) + mask | same as Ŷ | t0 + 10 … t0 + 60 |

B = batch of issuance times (windows) during training; B = 1 in operation.

**Alignment:**
- *Across sampling times:* everything is indexed by measurement time on the 10-min grid. Rain is summed per 10 min. Releases are step functions held at the last recorded value.
- *Across locations:* levels sit on their own nodes. Rain is assigned to nodes through A. Releases enter at Res. Res has no level, so its level and mask channels are 0, and the release channel is 0 at every other node.

**Available at issuance:** levels, rain and releases up to t0 (minus late readings), plus release schedules announced at or before t0. **Not available:** future rain, future levels, schedules announced after t0.

## 4. Model computations and design justification

**Notation:** x_{t,n} is the 4-feature input of node n at time t. h_n is the node's hidden state (size 64). up(n) is the set of nodes directly upstream of n. d_mn is the river distance from m to n. y_{t0,n} is the last observed level.

**1. Temporal encoder (same GRU for every node):**

  h_n = GRU(x_{t0−35, n}, …, x_{t0, n})

**2. Announced releases:** encode the next hour's schedule and add it to the reservoir node:

  e = MLP(future releases),  h_Res ← h_Res + e

**3. Spatial block: 4 message-passing layers (k = 1 … 4), downstream only:**

  h_n^(k) = ReLU( W_self · h_n^(k−1) + Σ_{m ∈ up(n)} W_up · [h_m^(k−1), d_mn] )

**4. Forecast mapping (predict the change from the last level):**

  ŷ_n = y_{t0, n} + MLP(h_n^(4)),  ŷ_n ∈ ℝ⁶ (the 6 horizons), for n = S1 … S4

**5. Training objective:** mean squared error over all horizons and stations, ignoring targets that are missing (m = 1 if observed):

  Loss = Σ m·(ŷ − y)² / Σ m $\rightarrow$ MSE Loss

**Why these choices:**
- **Graph along the river:** the main information is where water is coming from. Downstream edges let S2 see S1 and T1, S3 see S2, and so on.
- **4 layers:** the longest path Res → S1 → S2 → S3 → S4 has 4 hops, so the reservoir release can reach the S4 forecast. Fewer layers would cut it off.
- **GRU first, then graph:** each node's GRU state summarises its recent trend (rising or falling, how fast). Passing these summaries downstream lets the model learn the lag between stations without fixing a travel time. Because the state also reflects how high the river is, the learned lag can change with flow.
- **6 h window:** longer than the assumed travel time to S4, so upstream changes that will reach S4 within the hour are inside the window.
- **Direct 6-step output, not step-by-step:** no future observations exist to feed in, and feeding back predictions makes errors pile up.
- **Predicting the change:** the level an hour later is usually close to the current one, so the model only learns the change.
- **Small model (~60k parameters):** only 6 nodes, and floods are rare in the data, so a small model is less likely to overfit.

**How each driver reaches each forecast:**
- **Upstream conditions:** S1's state passes to S2, then S3, then S4 through the message-passing layers.
- **Monitored tributary T1:** a node feeding into S2, so it affects S2 and, through S2, S3 and S4.
- **Unmonitored tributary:** no direct input. It is covered indirectly by rainfall mapped near S3 (A11) and by S3's own recent levels. This is a limitation (Section 6).
- **Rainfall:** enters at the nodes it is mapped to (A9) and then moves downstream with the rest of the state.
- **Reservoir releases:** past releases enter through the Res node's GRU, and announced releases through e. Both start at Res and pass through S1 → S4.

**Physical assumptions vs. learned parts:**

| Physical / fixed (from facts and assumptions) | Learned from data |
|---|---|
| Graph structure and flow direction (A6) | How strongly each upstream node affects each downstream one (W_up, W_self) |
| River distances on the edges | Travel times and how they change with flow |
| Gauge-to-node rainfall map A (A9) | How rainfall turns into a rise in level |
| History window 6 h (A7) | How releases translate into downstream levels |

## 5. Operation and exceptional conditions

**(i) Missing observations**
- A missing level is filled with the last valid value of that station and its mask is set to 0. The model sees the mask, so it knows the value is a fill.
- If a station has been missing for more than 6 h, the fill is set to 0 (mask still 0), and the neighbouring stations carry the information.
- In training, 5–10 % of readings are dropped at random, so the model learns to cope with gaps.
- Missing targets are excluded from the loss (mask m).

**(ii) Delayed observations**
- The forecast is still issued at t0 using only what has arrived. A reading that has not arrived is treated as missing (mask 0), as in (i).
- Data are stored by measurement time, not arrival time. When the late reading arrives, it is used from the next issuance (10 min later). The forecast for t0 can also be re-issued if needed.

**(iii) Announced future releases**
- At t0, only schedules with announcement time ≤ t0 are used. Each announced value is placed in the future step (t0 + 10 … t0 + 60) matching its effective time.
- If two announcements cover the same time, the latest one announced before t0 is used.
- Steps with no announcement are filled with the last recorded release, with the announced flag set to 0.
- In training, schedules are used exactly as they were known at t0 (A10), never as known later.

**(iv) Scheduled vs. actual releases**
- The history channel always uses the **recorded actual** releases, never the schedule. Once the actual value for a time is recorded, it replaces the schedule in the next issuance.
- Because training uses the schedules as announced, the model learns how much it can trust them.
- The difference between scheduled and actual releases over the last hour is monitored. If it is large, the forecast is flagged as less reliable.

**Running every 10 minutes:** each run builds one window (6 nodes × 36 steps), applies one GRU pass, 4 small graph layers and an MLP, using about 60k parameters. This takes milliseconds on a normal CPU, far below the 10-minute interval. Training is done offline and repeated occasionally (e.g. monthly) as new data arrive.

## 6. Uncertainty, limitations, and additional information

**Uncertainty estimate:** no separate physical simulator or probabilistic model is used. Physics enters only through the graph, flow direction, distances and rain map (Section 4). To give a range, the forecast errors on the validation period are collected per station, per horizon and per water-level band (normal / high). Their 5th and 95th percentiles give a 90 % interval around each forecast.

**Limitations**

| Limitation | Effect on forecast | Response |
|---|---|---|
| Unmonitored tributary between S2 and S3 | Rises from it are seen late or underestimated at S3 and S4, especially after local rain. | Rain near S3 is a partial stand-in, and S3's own trend shows it once it starts. Wider intervals for S3/S4. Partly unresolved. |
| Changing travel times | If the lag is wrong, a flood wave can be forecast too early or too late. | The lag is learned from current levels (A8). Unresolved for flows beyond the training range. |
| No future rainfall | Rain falling within the next hour is not anticipated; mainly affects fast local responses. | None in the model; uncertainty is wider during rain. Unresolved without a rainfall forecast. |
| Few extreme floods in the data | Peaks may be underestimated, exactly when it matters most. | Separate high-water error bands give wider intervals. Partly unresolved. |
| Only two rain gauges | Rain in other parts of the catchment is missed. | Fixed map A; the error shows up in the intervals. Unresolved. |
| Scheduled vs. actual releases differ | The forecast below the reservoir can be off when operators deviate. | Actual releases replace the schedule as soon as they are recorded; deviations are flagged (Section 5). |
| Long sensor outages | Less information near that station. | Mask + neighbours (Section 5). Accuracy drops but the system keeps running. |
| Historical schedules not archived (if A10 fails) | Training cannot reproduce what was known at t0. | Use recorded releases as a stand-in during training; this makes the model trust schedules too much. |

**Additional information requests**

| Request | How it helps | Usable without it? |
|---|---|---|
| A level gauge on the unmonitored tributary | Adds it as a node feeding S3; removes the largest blind spot. | Yes, with wider S3/S4 intervals. |
| Rainfall forecasts or radar nowcasts for the next hour | Adds known-future rain, like the announced releases. | Yes. |
| Measured travel times or rating curves (level → discharge) | Checks the learned lags and helps at flows beyond the training range. | Yes. |
| Archive of past announced schedules with announcement times | Lets training use schedules exactly as known at t0 (A10). | Yes, using recorded releases as a stand-in. |
| Reporting delays and data-quality / maintenance logs | Sets realistic missing and delay handling; removes faulty data. | Yes, using A3–A5. |
| Record length and list of past flood events | Confirms enough data (A1) and how to weight floods. | Yes. |
| Catchment boundaries for R1 and R2 | Gives a better gauge-to-node map A. | Yes, using distance-based weights. |